In [1]:
import os
import re
import string
import random
import torch
from datasets import Dataset

### 1、下载模型

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ.pop('HTTP_PROXY', None)
os.environ.pop('HTTPS_PROXY', None)
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

model_name = "uer/gpt2-chinese-cluecorpussmall"
cache_dir = "./my_gpt2_model"

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=cache_dir)
print("模型下载完成")

D:\Software_local\conda\envs\nlp_lsh\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
D:\Software_local\conda\envs\nlp_lsh\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


模型下载完成


### 2、数据预处理

In [3]:
#读取原始诗集文件
with open('唐诗数据集NLP期末设计.txt', 'r', encoding='utf-8') as f:
    content = f.read()

#按[T]分割成每首诗
blocks = re.split(r'\[T\]', content)
poems = []

for block in blocks:
    block = block.strip()
    if not block:
        continue
    # 第一行是标题，后面是诗句
    lines = block.split('\n')
    title = lines[0].strip()
    text = ' '.join(lines[1:]).strip()
    # 判断是五言还是七言
    first_line = lines[1].strip() if len(lines) > 1 else ''
    # 去除标点符号后统计汉字数
    import string
    punct = '，。！？；：""''（）'
    clean = ''.join(c for c in first_line if c not in punct)
    if len(clean) == 20:
        genre = '五言诗'
    else:
        genre = '七言诗'  
    
    #构造训练样本：格式为 “体裁：标题 诗句”
    sample = f"{genre}：{title} {text}"
    poems.append(sample)

#保存为训练文件
with open('train_poems.txt', 'w', encoding='utf-8') as f:
    for p in poems:
        f.write(p + '\n')

print(f"共处理了 {len(poems)} 首诗")
print("第一条样本示例："+poems[0])

共处理了 50 首诗
第一条样本示例：五言诗：静夜思 床前明月光，疑是地上霜。举头望明月，低头思故乡。


In [4]:
with open('D:/python/python_5_nlp/train_poems.txt', 'r', encoding='utf-8') as f:
    poems = [line.strip() for line in f if line.strip()]

augmented = poems * 2
random.shuffle(augmented)  #打乱顺序

with open('train_poems_augmented.txt', 'w', encoding='utf-8') as f:
    for p in augmented:
        f.write(p + '\n')

print(f"数据量从 {len(poems)} 增加到 {len(augmented)}")

数据量从 50 增加到 100


### 3、微调前测试对比

In [5]:
#将模型设置为评估模式
model.eval()

#定义生成函数，用贪婪搜索
def generate_text(prompt, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_length=max_length,
            do_sample=False,   
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("------微调前生成结果------")

print("\n[测试1：开头为“床前明月光”] ")
print(generate_text("床前明月光，"))

print("\n[测试2：开头为“慈母手中线”] ")
print(generate_text("慈母手中线，"))

print("\n[测试3：“乱花渐欲迷人眼”] ")
print(generate_text("乱花渐欲迷人眼，"))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


------微调前生成结果------

[测试1：开头为“床前明月光”] 


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


床 前 明 月 光 ， 头 万 里 长 。 头 万 里 长 。 头 万 里 长 。 头 万 里 长 。 头 万 里 长 。 头 万 里 长 。 头 万 里 长 。

[测试2：开头为“慈母手中线”] 


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


慈 母 手 中 线 ，

[测试3：“乱花渐欲迷人眼”] 
乱 花 渐 欲 迷 人 眼 ， 《 花 样 姐 姐 》 第 二 季 第 二 期 ， 花 样 姐 姐 的 新 作 《 花 样 姐 姐 》 将 于 4 月 16 日 起 每 周 五 晚 22


### 4、微调模型

In [6]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

#加载训练数据
with open('train_poems_augmented.txt', 'r', encoding='utf-8') as f:
    lines = [line.strip() for line in f if line.strip()]
dataset = Dataset.from_dict({"text": lines})
print(f"数据集大小：{len(dataset)} 条")

#修复tokenizer的缺失标记
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.eos_token is None:
    tokenizer.eos_token = '</s>'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
#对文本进行分词
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

#设置数据整理器
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

#设置训练参数
training_args = TrainingArguments(
    output_dir="./poem_model_output_aug",
    overwrite_output_dir=True,
    num_train_epochs=8,
    per_device_train_batch_size=2,
    save_steps=100,
    logging_steps=50,
    learning_rate=5e-5,
    warmup_steps=50,
    save_total_limit=2,
    prediction_loss_only=True,
    report_to="none",
)

#创建Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset,
)

#开始训练
print("开始微调")
trainer.train()

model.save_pretrained("./my_finetuned_poem_model_aug")
tokenizer.save_pretrained("./my_finetuned_poem_model_aug")
print("模型已保存到 ./my_finetuned_poem_model_aug")

Using eos_token, but it is not set yet.


数据集大小：100 条


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

D:\Software_local\conda\envs\nlp_lsh\lib\site-packages\transformers\optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


开始微调


Step,Training Loss
50,3.902500
100,1.990000
150,0.847400
200,0.394900
250,0.260200
300,0.217300
350,0.181600
400,0.159500


模型已保存到 ./my_finetuned_poem_model_aug


###  5、微调后生成测试

In [9]:
#加载微调后的模型
finetuned_model = AutoModelForCausalLM.from_pretrained("./my_finetuned_poem_model_aug")
finetuned_model.eval()

#使用相同的生成函数，此处是贪婪搜素
def generate_finetuned(prompt, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = finetuned_model.generate(
            inputs.input_ids,
            max_length=max_length,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("====== 微调后生成结果（贪婪搜索）======")
print("\n[测试1] 床前明月光，")
print(generate_finetuned("床前明月光，"))

print("\n[测试2] 慈母手中线，")
print(generate_finetuned("慈母手中线，"))

print("\n[测试3] 乱花渐欲迷人眼，")
print(generate_finetuned("乱花渐欲迷人眼，"))

====== 微调后生成结果（贪婪搜索）======

[测试1] 床前明月光，
床 前 明 月 光 ， 十 八 楼 。 停 车 坐 爱 枫 林 晚 ， 霜 叶 红 于 二 月 花 。 古 来 征 战 几 人 回 。 葡 萄 美 酒 夜 光 杯 ， 欲 饮 琵 琶

[测试2] 慈母手中线，
慈 母 手 中 线 ， 周 手 中 金 。 慈 母 手 中 金 ， 温 润 如 玉 。 问 君 能 有 几 多 愁 ， 不 知 心 恨 谁 。 古 来 征 战 几 人 回 。 古

[测试3] 乱花渐欲迷人眼，
乱 花 渐 欲 迷 人 眼 ， 就 是 这 样 一 个 人 。 不 知 何 处 吹 芦 管 ， 一 夜 征 人 尽 望 乡 。 金 谷 园 繁 华 事 散 逐 香 尘 ， 流 水 无


### 6、调整参数进行试验

In [19]:
#定义一个灵活的实验函数
def experiment(prompt, do_sample=False, temperature=1.0, top_p=1.0, num_beams=1, max_length=60):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = finetuned_model.generate(
            inputs.input_ids,
            max_length=max_length,
            do_sample=do_sample,          #是否随机采样
            temperature=temperature,      #温度
            top_p=top_p,                  #Top-p采样（ do_sample=True）
            num_beams=num_beams,          #束搜索宽度（ num_beams>1 采用）
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 固定开头
prompt = "乱花渐欲迷人眼，"

print("1. 贪婪搜索 (do_sample=False)")
print(experiment(prompt, do_sample=False))

print("\n2. 束搜索 (num_beams=5)")
print(experiment(prompt, num_beams=5))

print("\n3. 随机采样，低温 (temperature=0.3)")
print(experiment(prompt, do_sample=True, temperature=0.3))

print("\n4. 随机采样，中温 (temperature=1.0)")
print(experiment(prompt, do_sample=True, temperature=1.0))

print("\n5. 随机采样，高温 (temperature=1.5)")
print(experiment(prompt, do_sample=True, temperature=1.5))

print("\n6. Top-p 采样 (top_p=0.9)")
print(experiment(prompt, do_sample=True, top_p=0.9))

1. 贪婪搜索 (do_sample=False)
乱 花 渐 欲 迷 人 眼 ， 就 是 这 样 一 个 人 。 不 知 何 处 吹 芦 管 ， 一 夜 征 人 尽 望 乡 。 金 谷 园 繁 华 事 散 逐 香 尘 ， 流 水 无 情 草 自 春 。 古 来 征 战

2. 束搜索 (num_beams=5)
乱 花 渐 欲 迷 人 眼 ， 最 是 一 年 春 好 处 。 近 乡 情 更 怯 ， 不 敢 问 来 人 。 古 来 征 战 几 人 回 ， 不 教 胡 马 度 阴 山 。 葡 萄 美 酒 夜 光 杯 ， 欲 饮 琵 琶

3. 随机采样，低温 (temperature=0.3)
乱 花 渐 欲 迷 人 眼 ， 就 是 这 样 一 β 胡 萝 卜 素 。 胡 萝 卜 素 是 维 生 素 a 的 重 要 来 源 ， 可 以 说 是 人 体 必 需 的 营 养 素 。 但 是 ， 胡 萝 卜 中 的 β 胡 萝

4. 随机采样，中温 (temperature=1.0)
乱 花 渐 欲 迷 人 眼 ， 春 来 发 几 枝 。 古 来 征 战 几 人 回 ， 不 教 胡 马 度 阴 山 。 葡 萄 美 酒 夜 光 杯 ， 欲 饮 琵 琶 马 上 催 。 葡 萄 美 酒 夜 光 杯 ， 欲

5. 随机采样，高温 (temperature=1.5)
乱 花 渐 欲 迷 人 眼 ， 车 成 泪 痕 相 照 两 千 里 路 ， 行 人 欲 断 魂 。 采 莲 曲 终 日 与 水 性 同 归 于 尽 ， 唯 见 窗 前 月 渐 寒 。 有 些 事 ， 你 始 终 记 不 起

6. Top-p 采样 (top_p=0.9)
乱 花 渐 欲 迷 人 眼 ， 工 欲 善 其 事 。 可 是 看 似 风 光 无 限 好 ， 可 是 看 似 光 鲜 背 后 隐 藏 的 苦 楚 总 是 很 多 。 有 时 候 ， 不 知 道 该 做 什 么 ， 才 能 让


### 7、补充实验：模型是否学会了根据提示生成特定体裁？

In [21]:
print("------ 微调前体裁控制测试 ------")
print("输入：五言诗")
print(generate_text("五言诗：", max_length=35))
print("\n输入：七言诗")
print(generate_text("七言诗：", max_length=40))

print("\n------ 微调后体裁控制测试 ------")
print("输入：五言诗：")
print(generate_finetuned("五言诗：", max_length=35))
print("\n输入：七言诗：")
print(generate_finetuned("七言诗：", max_length=40))

------ 微调前体裁控制测试 ------
输入：五言诗
五 言 诗 ： 有 静 夜 思 人 ， 静 夜 思 人 。 举 头 望 明 月 ， 低 头 思 故 乡 。 中 有 人 来 ，

输入：七言诗
七 言 诗 ： 十 九 日 忆 山 东 兄 弟 独 在 异 乡 为 异 客 ， 每 逢 佳 节 倍 思 亲 。 遥 知 兄 弟 登 高 处 ， 遍 插

------ 微调后体裁控制测试 ------
输入：五言诗：
五 言 诗 ： 上 受 降 城 闻 笛 回 乐 烽 前 沙 似 雪 ， 受 降 城 外 月 如 霜 。 不 知 何 处 吹 芦 管

输入：七言诗：
七 言 诗 ： 采 莲 曲 荷 叶 罗 裙 一 色 裁 ， 芙 蓉 向 脸 两 边 开 。 乱 入 池 中 看 不 见 ， 闻 歌 始 觉 有 人 来


- 微调后的模型**未能根据提示词切换体裁**。这个负面结果恰好印证了一个观点：**当前神经生成模型擅长学习词汇和短语的统计规律，但对符号化的“控制指令”响应较弱**，除非在大量数据中显式地、多样地训练。若要实现可靠的体裁控制，可能需要更大规模、更平衡的数据集，或者使用条件生成模型。